In [1]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
dataset_path = r"C:\Users\Shemeem\Downloads\archive (9)\signatures"

In [3]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 2112 images belonging to 2 classes.
Found 528 images belonging to 2 classes.


In [4]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

In [5]:
for layer in base_model.layers:
    layer.trainable = False

In [6]:
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)

output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

In [7]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 364s 4s/step - accuracy: 0.5232 - loss: 0.7106 - val_accuracy: 0.6080 - val_loss: 0.6874
Epoch 2/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 311s 5s/step - accuracy: 0.5672 - loss: 0.6801 - val_accuracy: 0.5568 - val_loss: 0.6820
Epoch 3/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 244s 4s/step - accuracy: 0.5682 - loss: 0.6737 - val_accuracy: 0.6004 - val_loss: 0.6780
Epoch 4/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 304s 4s/step - accuracy: 0.5663 - loss: 0.6704 - val_accuracy: 0.5322 - val_loss: 0.6899
Epoch 5/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 285s 4s/step - accuracy: 0.5895 - loss: 0.6616 - val_accuracy: 0.6345 - val_loss: 0.6775


In [9]:
loss, accuracy = model.evaluate(val_data)

print("Accuracy:", accuracy)

17/17 ━━━━━━━━━━━━━━━━━━━━ 54s 3s/step - accuracy: 0.6345 - loss: 0.6775 
Accuracy: 0.6344696879386902


In [23]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = r"C:\Users\Shemeem\Downloads\archive (9)\signatures\full_org\original_9_2.png"

img = image.load_img(img_path, target_size=(128, 128))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("Forged Signature")
else:
    print("Genuine Signature")

1/1 ━━━━━━━━━━━━━━━━━━━━ 28s 28s/step
Genuine Signature
